# Computer Vision 1st Homework
**Class:** Class02  
**Student ID:** 1705817  
**Name:** Yunsang Um

---
## 1. 현재까지 실습 코드 (DoG까지) 통합 및 결과 출력
각 과정에 대한 설명과 주석을 포함합니다.

### 실습 1: RGB → YUV 컬러 모델 변환
BGR 이미지를 YUV 컬러 공간으로 변환하고 각 채널을 분리하여 확인합니다.

In [ ]:
import cv2
import numpy as np
import urllib.request
from google.colab.patches import cv2_imshow

# 1. 이미지 다운로드 및 로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/smarties.png'
urllib.request.urlretrieve(url, 'smarties.png')
img = cv2.imread('smarties.png')

# 2. BGR에서 YUV로 컬러 공간 변환
yuv_img = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)

# 3. 채널별로 분리 (Y: 밝기, U: 청색편차, V: 적색편차)
y, u, v = cv2.split(yuv_img)

# 4. 결과 시각화 (가로로 결합)
print("--- 실습 1 결과: RGB to YUV ---")
cv2_imshow(np.hstack((y, u, v)))

### 실습 2: Gamma Correction (감마 조절)
룩업 테이블을 사용하여 영상의 밝기를 비선형적으로 조절합니다.

In [ ]:
def adjust_gamma(image, gamma=1.0):
    # 감마 보정을 위한 룩업 테이블(LUT) 생성
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/fruits.jpg'
urllib.request.urlretrieve(url, 'fruits.jpg')
img = cv2.imread('fruits.jpg')

# 감마 값 적용 (밝게: 2.2, 어둡게: 0.4)
res = np.hstack((img, adjust_gamma(img, 2.2), adjust_gamma(img, 0.4)))
print("--- 실습 2 결과: Gamma Correction (원본 | 2.2 | 0.4) ---")
cv2_imshow(res)

### 실습 3: 히스토그램 평활화 (Histogram Equalization)
명암 대비가 낮은 영상의 히스토그램을 균일하게 분포시켜 화질을 개선합니다.

In [ ]:
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'
urllib.request.urlretrieve(url, 'vtest.avi')
cap = cv2.VideoCapture('vtest.avi')

ret, frame = cap.read() # 첫 프레임만 예시로 수행
if ret:
    # 1. YUV 변환 후 Y(밝기) 채널 평활화
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y, u, v = cv2.split(yuv)
    y_eq = cv2.equalizeHist(y)
    
    # 2. 다시 BGR로 복구
    res_frame = cv2.cvtColor(cv2.merge([y_eq, u, v]), cv2.COLOR_YUV2BGR)
    
    print("--- 실습 3 결과: Histogram Equalization (원본 | 결과) ---")
    cv2_imshow(np.hstack((frame, res_frame)))
cap.release()

### 실습 4: Convolution (Filtering)
엠보싱, 샤프닝, 평균 필터를 적용하여 영상을 처리합니다.

In [ ]:
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/baboon.jpg'
urllib.request.urlretrieve(url, 'baboon.jpg')
img = cv2.imread('baboon.jpg')

# 필터 커널 정의
emboss = np.array([[-1, -1, 0], [-1, 0, 1], [0, 1, 1]])
sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])

print("--- 실습 4 결과: Convolution Filters (엠보싱 | 샤프닝) ---")
cv2_imshow(np.hstack((cv2.filter2D(img, -1, emboss)+128, cv2.filter2D(img, -1, sharp))))

### 실습 5: Gaussian Filtering
가우시안 분포를 이용한 블러 처리를 수행합니다.

In [ ]:
blur_img = cv2.GaussianBlur(img, (9, 9), 0)
print("--- 실습 5 결과: Gaussian Blur (9x9) ---")
cv2_imshow(blur_img)

### 실습 6 & 7: Edge Detection (Sobel & Canny)
경계선 검출 알고리즘을 적용합니다.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Sobel
sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel = cv2.convertScaleAbs(cv2.addWeighted(sx, 0.5, sy, 0.5, 0))

# Canny
canny = cv2.Canny(gray, 100, 200)

print("--- 실습 6, 7 결과: Sobel vs Canny ---")
cv2_imshow(np.hstack((sobel, canny)))

### 실습 8: Morphology (모폴로지)
침식과 팽창 연산을 통해 형태학적 처리를 수행합니다.

In [ ]:
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = np.ones((5, 5), np.uint8)
erosion = cv2.erode(binary, kernel, iterations=1)
dilation = cv2.dilate(binary, kernel, iterations=1)

print("--- 실습 8 결과: Morphology (이진화 | 침식 | 팽창) ---")
cv2_imshow(np.hstack((binary, erosion, dilation)))

### 실습 9 & 10: Interpolation & Geometric Transform
영상 보간법과 기본적인 기하 변환을 수행합니다.

In [ ]:
rows, cols = img.shape[:2]
# 이동 및 회전 (OpenCV 내장 함수 사용 예시)
M = cv2.getRotationMatrix2D((cols/2, rows/2), 45, 0.7)
dst = cv2.warpAffine(img, M, (cols, rows))

print("--- 실습 9, 10 결과: Rotation and Scale (OpenCV) ---")
cv2_imshow(dst)

### 실습 11: SIFT DoG Pyramid 생성
가우시안 피라미드의 차분(DoG)을 통해 스케일 공간을 생성합니다.

In [ ]:
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg'
urllib.request.urlretrieve(url, 'butterfly.jpg')
bf_img = cv2.imread('butterfly.jpg', 0)

s = 3
k = 2**(1.0/s)
g1 = cv2.GaussianBlur(bf_img, (0, 0), 1.6)
g2 = cv2.GaussianBlur(bf_img, (0, 0), 1.6 * k)
dog = cv2.subtract(g2.astype(np.float32), g1.astype(np.float32))
dog_norm = cv2.normalize(dog, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

print("--- 실습 11 결과: Difference of Gaussian (Sample) ---")
cv2_imshow(dog_norm)

---
## 2. [HW1] Butterfly 영상 기하 변환 (회전 및 이동)
**요구사항:**
- **동치좌표계(Homogeneous Coordinates)**를 이용한 변환 행렬 구성
- 중심을 축으로 **225도 회전** 및 **(80, 80) 이동**
- **Forward Mapping** 우선 수행 후, 빈 공간(Hole)은 **Backward Mapping**으로 보간
- 픽셀 단위로 직접 수행 (고수준 변환 함수 사용 금지)
- 변환 공간을 충분히 넓게 잡아 영상이 잘리지 않도록 함

In [ ]:
def advanced_geometric_transform(img, angle_deg, tx, ty):
    # 1. 기본 정보 및 행렬 설정
    h, w = img.shape[:2]
    rad = np.radians(angle_deg)
    
    # 결과 영상 크기 설정 (넉넉하게 설정)
    diag = int(np.sqrt(w**2 + h**2))
    out_size = diag + 200
    out_img = np.zeros((out_size, out_size, 3), dtype=np.uint8)
    filled_mask = np.zeros((out_size, out_size), dtype=np.uint8) # 채워진 픽셀 체크용
    
    # 동치좌표계 행렬 정의
    # (1) 중심을 원점으로 이동
    T1 = np.array([[1, 0, -w/2], [0, 1, -h/2], [0, 0, 1]])
    # (2) 회전 행렬
    R = np.array([[np.cos(rad), -np.sin(rad), 0], 
                  [np.sin(rad), np.cos(rad), 0], 
                  [0, 0, 1]])
    # (3) 결과 영상의 중심으로 이동 및 과제 조건 (80, 80) 이동
    T2 = np.array([[1, 0, out_size/2 + tx], 
                  [0, 1, out_size/2 + ty], 
                  [0, 0, 1]])
    
    # 최종 변환 행렬 H = T2 * R * T1
    H = T2 @ R @ T1
    H_inv = np.linalg.inv(H) # Backward mapping용 역행렬

    # 2. Forward Mapping 수행
    print("Forward Mapping 수행 중...")
    for y in range(h):
        for x in range(w):
            p = np.array([x, y, 1])
            p_new = H @ p
            nx, ny = int(p_new[0]), int(p_new[1])
            
            if 0 <= nx < out_size and 0 <= ny < out_size:
                out_img[ny, nx] = img[y, x]
                filled_mask[ny, nx] = 1

    # 3. Backward Mapping으로 빈 공간(Hole) 채우기
    print("Backward Mapping으로 빈 공간 보간 중...")
    for y in range(out_size):
        for x in range(out_size):
            if filled_mask[y, x] == 0: # Forward mapping으로 채워지지 않은 곳
                p_dst = np.array([x, y, 1])
                p_src = H_inv @ p_dst
                sx, sy = p_src[0], p_src[1]
                
                # 원본 영상 범위 내에 있다면 보간(Bilinear)
                if 0 <= sx < w-1 and 0 <= sy < h-1:
                    x0, y0 = int(sx), int(sy)
                    dx, dy = sx - x0, sy - y0
                    
                    pixel = (1-dx)*(1-dy)*img[y0, x0] + dx*(1-dy)*img[y0, x0+1] + \
                            (1-dx)*dy*img[y0+1, x0] + dx*dy*img[y0+1, x0+1]
                    out_img[y, x] = pixel.astype(np.uint8)
                    
    return out_img

# Butterfly 이미지 로드
butterfly = cv2.imread('butterfly.jpg')

# 과제 수행: 225도 회전 및 (80, 80) 이동
hw_result = advanced_geometric_transform(butterfly, 225, 80, 80)

print("\n--- [HW1 결과] Butterfly: 225도 회전 및 (80, 80) 이동 (Forward + Backward Hybrid) ---")
cv2_imshow(hw_result)